In [ ]:
!pip install -q transformers peft librosa soundfile sinling torch fastapi uvicorn pyngrok python-multipart nest-asyncio joblib scikit-learn "torchao>=0.16.0"

In [ ]:
import torch, librosa, numpy as np, time, os
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor,
    Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification,
)
from peft import PeftModel
from sinling import SinhalaTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
# fp16 roughly halves Whisper's inference time on a GPU and costs no accuracy
# that matters here. CPU stays fp32 — half precision is slower on CPU.
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}  dtype: {dtype}")
if device == "cpu":
    print("WARNING: no GPU. Whisper-medium on CPU is ~10-20x slower.")
    print("         In Colab: Runtime -> Change runtime type -> T4 GPU.")

# ── Whisper ASR ──
BASE_MODEL = "openai/whisper-medium"
ADAPTER_ID = "SPEAK-ASR/whisper-si-exp-10-medium-all"

whisper_processor = WhisperProcessor.from_pretrained(BASE_MODEL)
whisper_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
whisper_model = PeftModel.from_pretrained(whisper_model, ADAPTER_ID)
whisper_model = whisper_model.merge_and_unload().to(device=device, dtype=dtype).eval()
print("Whisper ready")

# ── Sinhala tokenizer ──
# POS tagging was removed: it ran on every request and the web app never
# received the result (app/api/stt/route.ts drops pos_tags), so it was pure
# latency. Only the tokenizer is needed.
sinhala_tokenizer = SinhalaTokenizer()
print("Tokenizer ready")

# ── Feature extraction ──
# Injected verbatim from notebook-integration/_shared_features.py, the same
# source EmotionClassifier.ipynb is built from. Sharing one definition is what
# prevents train/serve skew: if the two computed features even slightly
# differently, the classifier would receive vectors that mean something other
# than what it learned on, and accuracy would degrade with nothing in the logs.
# scipy ships with librosa, so this adds no dependency.
from scipy.signal import medfilt

# ── Framing ──
# Shared by pitch and energy so the two line up frame-for-frame and the voicing
# gate can index one against the other.
SAMPLE_RATE = 16000
FRAME_LENGTH = 1024
HOP_LENGTH = 256
# Human speech fundamental.
#
# The floor is 55 Hz, not the more common 65: a deep male voice speaking quietly
# — which is what sad delivery sounds like — drops into the 50s, and anything
# below the floor is discarded as unvoiced rather than measured. Set at 65, a
# low-pitched speaker loses precisely the frames that carry the emotion, and
# every pitch statistic is then computed from whatever survived.
#
# The ceiling stays at 400: above that is singing, not speech.
#
# This range is also why the pipeline is fast. librosa's own examples reach for
# `pyin` over C2-C7 (65-2093 Hz), which is a *music* range; measured on a 2 s
# clip that costs ~570 ms, about 90% of the whole feature pipeline. `yin` over
# the speech range does the same job in ~3 ms.
F0_MIN, F0_MAX = 55.0, 400.0


def _safe(x, default=0.0):
    """Feature values must be finite — sklearn refuses NaN, and one bad clip
    would otherwise take down the whole fit."""
    x = float(x) if x is not None else default
    return x if np.isfinite(x) else default


def _slope(values):
    """Linear trend over time. Falling pitch reads as sad or resigned; rising
    reads as surprised or questioning."""
    if len(values) < 2:
        return 0.0
    t = np.arange(len(values), dtype=np.float64)
    return _safe(np.polyfit(t, values, 1)[0])


# Longest silence kept between two bursts of speech, in seconds.
MAX_GAP_S = 0.35


def compact_speech(waveform, sr=SAMPLE_RATE, top_db=30, max_gap_s=MAX_GAP_S):
    """
    Keep the speech, cap the gaps between it.

    A recording is two things mixed together: how someone spoke, and how the
    recording was made. A fixed four-second capture window where the speaker
    talks for one and a half seconds is 60% dead air — and that dead air moves
    `silence_ratio`, `voiced_ratio`, `rms_cv`, `mfcc0_mean` and the onset-gap
    features by three to twenty-nine standard deviations. Set against UrduSER,
    which is tightly-cut broadcast dialogue, the two look like different
    domains when the only real difference is where somebody pressed stop.

    Trailing silence is pure artefact and goes entirely. Internal pauses are
    real prosody — sad speech genuinely pauses more — so they are kept but
    capped, which preserves *that a pause happened* and *roughly how many*
    without letting one long think-pause dominate every statistic.

    Returns the original if nothing above the threshold is found, so a quiet
    clip degrades rather than becoming empty.
    """
    if waveform.size == 0:
        return waveform
    intervals = librosa.effects.split(waveform, top_db=top_db)
    if len(intervals) == 0:
        return waveform

    max_gap = int(max_gap_s * sr)
    pieces, previous_end = [], None
    for start, end in intervals:
        if previous_end is not None:
            gap = min(start - previous_end, max_gap)
            if gap > 0:
                pieces.append(waveform[previous_end:previous_end + gap])
        pieces.append(waveform[start:end])
        previous_end = end

    compacted = np.concatenate(pieces) if pieces else waveform
    return compacted if compacted.size else waveform


def extract_features(waveform, sr=SAMPLE_RATE):
    """
    One clip -> one fixed-length vector of prosodic and spectral statistics.

    Each block maps to something a listener actually hears as emotion:

      pitch (F0)     : angry and happy sit higher and vary more; sad is flat
      energy (RMS)   : angry is loud with sharp attacks; sad is quiet
      rhythm         : sad is slow with long pauses; angry is fast and clipped
      voice quality  : spectral shape separates tense from breathy
      MFCC           : timbre, the general-purpose backstop

    Deliberately *not* normalised per speaker: doing so would leak test-speaker
    statistics into training under a speaker-independent split.

    ## Why the waveform is peak-normalised first

    Without it, `rms_mean`, `rms_max`, `rms_range`, `rms_std`, `rms_slope` and
    `mfcc0_mean` all scale with the raw recording level — measured, a x0.15
    gain change moves them by about 85%. Those were also the six features the
    model ranked most important, so most of its decision was really "how loud
    is this file".

    That is fine within one corpus, where the recording chain is constant and
    loudness genuinely correlates with anger. It is fatal across corpora:
    UrduSER is broadcast television audio, normalised and compressed to a peak
    near 1.0, while a laptop microphone is far quieter. A model trained on the
    former and served the latter reads every clip as low-energy and scores
    *below chance* — which is exactly what happened.

    Normalising to unit peak throws away absolute level and keeps the shape:
    how much the energy varies, where it rises and falls, how peaky it is
    against its own maximum. That is the part that actually carries emotion,
    and it survives a change of microphone.

    Done here rather than in `load_and_trim` on purpose — the serving path
    passes an already-decoded waveform straight to this function, so anything
    done outside it would apply during training and not during serving.
    """
    # Strip the recording protocol before measuring the speech: cap dead air,
    # then remove absolute level. See compact_speech() and the note above.
    waveform = compact_speech(waveform, sr)
    peak = float(np.max(np.abs(waveform))) if waveform.size else 0.0
    if peak > 1e-6:
        waveform = waveform / peak

    feats, names = [], []

    def add(name, value):
        names.append(name)
        feats.append(_safe(value))

    # ── energy (first: the voicing gate below needs it) ──
    rms = librosa.feature.rms(y=waveform, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]

    # ── pitch ──
    f0 = librosa.yin(waveform, fmin=F0_MIN, fmax=F0_MAX, sr=sr,
                     frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)
    # yin reports a pitch for every frame including silence, so voiced frames
    # have to be found separately: loud enough to be speech, and with a pitch
    # not pinned to the edge of the search range (where yin lands when there is
    # nothing periodic to find).
    n_frames = min(len(f0), len(rms))
    f0, rms = f0[:n_frames], rms[:n_frames]
    loud_enough = rms > max(0.10 * float(np.max(rms)) if rms.size else 0.0, 1e-4)
    in_band = (f0 > F0_MIN * 1.02) & (f0 < F0_MAX * 0.98)
    voiced_flag = loud_enough & in_band
    voiced = f0[voiced_flag]

    # Pitch is described *relative to the speaker's own register*, never in
    # absolute hertz.
    #
    # Absolute F0 is mostly a fact about the speaker's body, not their mood.
    # UrduSER averages 200 Hz; a male Sinhala speaker sits near 100 Hz — about
    # two standard deviations below the corpus mean before he has expressed
    # anything at all. A model given raw hertz learns "low pitch means sad" and
    # then labels that speaker sad no matter how he performs.
    #
    # Dividing by the clip's own median keeps the part that carries emotion —
    # how far the voice ranges, how much it varies, which way it drifts — and
    # discards the part that is just anatomy. Standard practice in
    # cross-speaker emotion work, and the whole reason this can cross a
    # language boundary.
    # Everything below is measured in **semitones away from this clip's own
    # median pitch**, which makes it register-free by construction: scaling a
    # voice up or down an octave shifts every value by the same constant, and
    # subtracting the median removes it exactly.
    #
    # The contour is median-filtered first. `yin` occasionally reports a pitch
    # an octave out on a single frame, and while that barely moves a percentile
    # it badly corrupts anything built from extremes, standard deviations or
    # frame-to-frame differences — measured, those moved 27-63% under an octave
    # shift purely from which frames happened to glitch. A 5-frame median
    # filter removes the spikes without smoothing real intonation, which
    # changes over far longer spans.
    #
    # Spread is then reported from percentiles rather than min/max for the same
    # robustness reason.
    if voiced.size >= 2:
        smooth = medfilt(voiced, kernel_size=5) if voiced.size >= 5 else voiced
        median = float(np.median(smooth))
        semitones = 12.0 * np.log2(np.maximum(smooth, 1e-6) / max(median, 1e-6))
        p10, p25, p75, p90 = np.percentile(semitones, [10, 25, 75, 90])
        add("f0_st_std", np.std(semitones))
        add("f0_st_iqr", p75 - p25)
        add("f0_st_range", p90 - p10)
        add("f0_st_high", p90)
        add("f0_st_low", p10)
        add("f0_st_slope", _slope(semitones))
        # Frame-to-frame pitch jitter is deliberately NOT a feature. It is a
        # real voice-quality cue, but it lives at exactly the timescale that
        # lossy codecs alter: browser capture is Opus, UrduSER came off
        # YouTube, and the two encode micro-variation differently. Measured, it
        # was the only pitch feature still moving more than half a standard
        # deviation under a register change. A feature that tracks the codec
        # more than the speaker is worse than no feature.
        #
        # The one absolute value, kept for reporting. It is the speaker's
        # register — anatomy, not mood — and the model is free to ignore it.
        add("f0_median_hz", median)
    else:
        for n in ["f0_st_std","f0_st_iqr","f0_st_range","f0_st_high","f0_st_low",
                  "f0_st_slope","f0_median_hz"]:
            add(n, 0.0)
    add("voiced_ratio", np.mean(voiced_flag) if voiced_flag.size else 0.0)

    # ── energy statistics ──
    add("rms_mean", np.mean(rms))
    add("rms_std", np.std(rms))
    add("rms_max", np.max(rms))
    add("rms_range", np.ptp(rms))
    add("rms_slope", _slope(rms))
    add("rms_cv", np.std(rms) / max(np.mean(rms), 1e-6))

    # ── rhythm ──
    duration = len(waveform) / sr
    add("duration", duration)
    # Silence share and pause rate stand in for speaking rate without needing a
    # word count. Sad speech pauses more, and for longer.
    threshold = 0.15 * np.mean(rms) if np.mean(rms) > 0 else 0.0
    quiet = rms < threshold
    add("silence_ratio", np.mean(quiet))
    add("pause_count", int(np.sum(np.abs(np.diff(quiet.astype(int))))) / max(duration, 1e-6))
    # Onsets per second: a direct proxy for syllable rate.
    onsets = librosa.onset.onset_detect(y=waveform, sr=sr, units="time")
    add("onset_rate", len(onsets) / max(duration, 1e-6))
    if len(onsets) > 1:
        gaps = np.diff(onsets)
        add("onset_gap_mean", np.mean(gaps))
        add("onset_gap_std", np.std(gaps))
    else:
        add("onset_gap_mean", 0.0)
        add("onset_gap_std", 0.0)

    # ── voice quality ──
    add("centroid_mean", np.mean(librosa.feature.spectral_centroid(y=waveform, sr=sr)[0]))
    add("centroid_std", np.std(librosa.feature.spectral_centroid(y=waveform, sr=sr)[0]))
    rolloff = librosa.feature.spectral_rolloff(y=waveform, sr=sr)[0]
    add("rolloff_mean", np.mean(rolloff))
    add("rolloff_std", np.std(rolloff))
    add("bandwidth_mean", np.mean(librosa.feature.spectral_bandwidth(y=waveform, sr=sr)[0]))
    zcr = librosa.feature.zero_crossing_rate(waveform)[0]
    add("zcr_mean", np.mean(zcr))
    add("zcr_std", np.std(zcr))
    add("flatness_mean", np.mean(librosa.feature.spectral_flatness(y=waveform)[0]))

    # ── timbre ──
    mfcc = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13)
    for i in range(13):
        add(f"mfcc{i}_mean", np.mean(mfcc[i]))
        add(f"mfcc{i}_std", np.std(mfcc[i]))
    # Deltas capture *change* in timbre — articulation sharpness.
    delta = librosa.feature.delta(mfcc)
    for i in range(13):
        add(f"mfcc{i}_delta_mean", np.mean(np.abs(delta[i])))

    return np.array(feats, dtype=np.float64), names


def load_and_trim(path, sr=SAMPLE_RATE):
    """Decode, downmix, resample, and strip leading/trailing silence.

    Trimming matters more than it looks: recordings carry dead air from the
    record button, and `silence_ratio` and `duration` would otherwise measure
    the operator's reflexes rather than the speaker's delivery. UrduSER ships
    at 44.1 kHz; librosa resamples to 16 kHz here so training and serving see
    identical input.
    """
    waveform, _ = librosa.load(path, sr=sr, mono=True)
    waveform, _ = librosa.effects.trim(waveform, top_db=30)
    return waveform.astype(np.float32)

# ── Emotion ──
# Two paths, decided by whether the trained classifier is present.
#
#   emotion_clf.joblib  -> the prosody model from EmotionClassifier.ipynb.
#                          Trained on the four emotions the avatar performs,
#                          and ~20x faster than the transformer below.
#   otherwise           -> the off-the-shelf English wav2vec2 model, so the
#                          notebook still runs before the classifier exists.
#
# Automatic rather than a flag: forgetting to flip a switch after uploading
# the artifact would silently keep serving the worse model.
EMO_CLF_PATH = "/content/emotion_clf.joblib"
USE_TRAINED_EMOTION = os.path.exists(EMO_CLF_PATH)

if USE_TRAINED_EMOTION:
    import joblib
    _emo_bundle = joblib.load(EMO_CLF_PATH)
    emo_model = _emo_bundle["model"]
    EMO_LABELS = _emo_bundle["labels"]
    _m = _emo_bundle["metrics"]
    print(f"Emotion classifier ready — {_m['algorithm']} trained on {_m.get('trained_on','?')}, "
          f"macro-F1 {_m['macro_f1']*100:.1f}% ({_m['cv']})")
else:
    EMO_ID = "r-f/wav2vec-english-speech-emotion-recognition"
    emo_extractor = Wav2Vec2FeatureExtractor.from_pretrained(EMO_ID)
    emo_model = Wav2Vec2ForSequenceClassification.from_pretrained(EMO_ID).to(device).eval()
    print("Emotion classifier ready — off-the-shelf English wav2vec2 (fallback)")
    print(f"  Upload {EMO_CLF_PATH} and re-run this cell to use the trained model.")

# ── Warm-up ──
# The first inference pays CUDA kernel compilation and lazy weight init, which
# can be 5-10s. Doing it here means the first *real* request doesn't.
_silence = np.zeros(16000, dtype=np.float32)
with torch.inference_mode():
    _f = whisper_processor(_silence, sampling_rate=16000, return_tensors="pt")
    whisper_model.generate(
        _f.input_features.to(device=device, dtype=dtype),
        language="sinhala", task="transcribe", max_new_tokens=8, num_beams=1,
    )
    if not USE_TRAINED_EMOTION:
        emo_model(**emo_extractor(_silence, sampling_rate=16000, return_tensors="pt").to(device))
# librosa and numba compile on first call too — a few hundred ms that would
# otherwise land on the first real request.
if USE_TRAINED_EMOTION:
    _v, _ = extract_features(np.random.randn(16000).astype(np.float32) * 0.01)
    emo_model.predict_proba([_v])
print("Warm-up done — models are hot\n")

In [ ]:
# Cap on generated tokens. Whisper defaults to 448, and without a limit a
# short or silent clip can ramble until it hits that ceiling — the single
# biggest source of unpredictable latency. A sign-language utterance is short.
MAX_NEW_TOKENS = 96

def load_audio(path):
    """Decode once, reuse for both models. The old code loaded the file twice."""
    waveform, _ = librosa.load(path, sr=16000, mono=True)
    return waveform.astype(np.float32)

def transcribe(waveform):
    inputs = whisper_processor(waveform, sampling_rate=16000, return_tensors="pt")
    features = inputs.input_features.to(device=device, dtype=dtype)
    with torch.inference_mode():
        ids = whisper_model.generate(
            features,
            language="sinhala",
            task="transcribe",
            max_new_tokens=MAX_NEW_TOKENS,
            # Greedy. Beam search multiplies decoder passes for accuracy we
            # don't need on short utterances.
            num_beams=1,
        )
    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

def detect_emotion(waveform):
    """
    Returns {"emotion": str, "confidence": float|None}.

    This shape is the contract with the web app — app/api/stt/route.ts reads
    `emotion.emotion` and `emotion.confidence`, and lib/emotion/styles.ts maps
    the label onto an avatar posture. Both branches below honour it, so
    swapping the model changes nothing on the web side.
    """
    if USE_TRAINED_EMOTION:
        # Trim first: the model was trained on trimmed audio, and leading
        # silence would skew duration and pause features away from anything
        # it has seen.
        trimmed, _ = librosa.effects.trim(waveform, top_db=30)
        if trimmed.size < 0.4 * 16000:
            return {"emotion": "neutral", "confidence": None}
        vec, _ = extract_features(trimmed)
        probs = emo_model.predict_proba([vec])[0]
        i = int(np.argmax(probs))
        return {"emotion": EMO_LABELS[i], "confidence": round(float(probs[i]), 3)}

    inputs = emo_extractor(waveform, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.inference_mode():
        logits = emo_model(**inputs.to(device)).logits
    scores = torch.nn.functional.softmax(logits, dim=1)[0]
    idx = int(torch.argmax(scores))
    return {"emotion": emo_model.config.id2label[idx], "confidence": round(float(scores[idx]), 3)}

def full_pipeline(audio_path, want_emotion=True):
    """
    Audio -> transcript + Sinhala word tokens.

    Note what is NOT here any more: the Sinhala->gloss lookup. It used to be a
    dict here doing exact matches, so any inflected or misspelled word
    ("කොහෙදද" for "කොහෙද") was silently dropped and the sign was lost.

    Mapping now happens in the web app (lib/nlp/matchers/sinhalaMatcher.ts),
    which handles suffix stripping and sound-alike spellings, and reads one
    dictionary from MongoDB that is editable from Dashboard -> Animations.
    Two dictionaries that could drift apart is now one.
    """
    t0 = time.perf_counter()
    waveform = load_audio(audio_path)
    t_load = time.perf_counter()

    text = transcribe(waveform)
    t_asr = time.perf_counter()

    tokens = sinhala_tokenizer.tokenize(text) if text else []
    t_tok = time.perf_counter()

    emotion = detect_emotion(waveform) if want_emotion else {"emotion": "neutral", "confidence": None}
    t_emo = time.perf_counter()

    return {
        "transcription": text,
        "tokens": tokens,
        "emotion": emotion,
        # Kept for older clients; the app prefers `tokens`.
        "glosses": [],
        "unknown_tokens": [],
        "timings_ms": {
            "audio": round((t_load - t0) * 1000),
            "asr": round((t_asr - t_load) * 1000),
            "tokenize": round((t_tok - t_asr) * 1000),
            "emotion": round((t_emo - t_tok) * 1000),
            "total": round((t_emo - t0) * 1000),
        },
    }

print("Pipeline ready")

In [ ]:
from fastapi import FastAPI, UploadFile, File, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import uvicorn, tempfile, os, threading, getpass
from pyngrok import ngrok

app = FastAPI(title="SSL Pipeline API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def health():
    return {"status": "ok", "device": device, "dtype": str(dtype)}

@app.post("/translate")
async def translate(audio: UploadFile = File(...), emotion: bool = Query(True)):
    """
    WAV/WebM in, transcript + Sinhala tokens out.

    `?emotion=false` skips the second model pass when the caller doesn't need
    the emotion badge — worth a few hundred ms on CPU.
    """
    suffix = os.path.splitext(audio.filename or "")[1] or ".wav"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(await audio.read())
        tmp_path = tmp.name
    try:
        result = full_pipeline(tmp_path, want_emotion=emotion)
        print("timings:", result["timings_ms"])
        return JSONResponse(content=result)
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})
    finally:
        os.unlink(tmp_path)

# ── ngrok ──
# Read the token instead of hardcoding it. A token committed to a notebook is
# a live credential anyone with the file can use.
ngrok.kill()
token = os.environ.get("NGROK_AUTHTOKEN") or getpass.getpass("ngrok authtoken: ")
ngrok.set_auth_token(token)

public_url = ngrok.connect(8000)
print(f"\n{'='*60}\nPUBLIC API URL: {public_url}\n{'='*60}")
print("Paste this into Dashboard -> Settings.\n")

def run_server():
    import asyncio
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000)).serve())

threading.Thread(target=run_server, daemon=True).start()
print("Server running.")

In [ ]:
# Optional: measure where the time actually goes, on a real clip.
# Upload a short wav to the Colab file browser and set the path.
TEST_AUDIO = "/content/sample.wav"

import os
if os.path.exists(TEST_AUDIO):
    for i in range(3):
        r = full_pipeline(TEST_AUDIO)
        print(f"run {i+1}: {r['timings_ms']}  ->  {r['transcription'][:60]}")
    print("\nFirst run may be slower; after that it is steady state.")
else:
    print(f"No file at {TEST_AUDIO} — upload one to benchmark.")